# Интеллектуальный анализ данных – весна 2025
# Домашнее задание 6: классификация текстов

Правила:



*   Домашнее задание оценивается в 10 баллов.
*   Можно использовать без доказательства любые результаты, встречавшиеся на лекциях или семинарах по курсу, если получение этих результатов не является вопросом задания.
*  Можно использовать любые свободные источники с *обязательным* указанием ссылки на них.
*  Плагиат не допускается. При обнаружении случаев списывания, 0 за работу выставляется всем участникам нарушения, даже если можно установить, кто у кого списал.
*  Старайтесь сделать код как можно более оптимальным. В частности, будет штрафоваться использование циклов в тех случаях, когда операцию можно совершить при помощи инструментов библиотек, о которых рассказывалось в курсе.
* Если в задании есть вопрос на рассуждение, то за отсутствие ответа на него балл за задание будет снижен вполовину.

В этом домашнем задании вам предстоит построить классификатор текстов.

Будем предсказывать эмоциональную окраску твиттов о коронавирусе.



In [92]:
import numpy as np
import pandas as pd
from typing import  List
import matplotlib.pyplot as plt
import seaborn as sns
from string import punctuation

In [93]:
df = pd.read_csv('https://github.com/hse-ds/iad-intro-ds/raw/master/2025/homeworks/hw06_texts/tweets_coronavirus.csv', encoding='latin-1')
df.sample(4)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
28168,38343,83295,London,08-04-2020,What will the post-COVID-19 consumer look like...,Positive
19874,28015,72967,"Baltimore, MD",26-03-2020,It's crazy how going to the grocery store is s...,Extremely Negative
31702,42798,87750,"Tokyo-to, Japan",12-04-2020,Best online stores to buy things from Tokyo an...,Extremely Positive
5110,10012,54964,Ohio,19-03-2020,#AngelaMerkel Nails #Coronavirus Speech ÃÂ U...,Positive


Для каждого твитта указано:


*   UserName - имя пользователя, заменено на целое число для анонимности
*   ScreenName - отображающееся имя пользователя, заменено на целое число для анонимности
*   Location - местоположение
*   TweetAt - дата создания твитта
*   OriginalTweet - текст твитта
*   Sentiment - эмоциональная окраска твитта (целевая переменная)



## Задание 1 Подготовка (0.5 балла)

Целевая переменная находится в колонке `Sentiment`.  Преобразуйте ее таким образом, чтобы она стала бинарной: 1 - если у твитта положительная или очень положительная эмоциональная окраска и 0 - если отрицательная или очень отрицательная.

In [94]:
df['Sentiment'].unique()

array(['Positive', 'Extremely Negative', 'Negative', 'Extremely Positive'],
      dtype=object)

In [95]:
positive_raw = (df['Sentiment'] == 'Positive') | (df['Sentiment'] == 'Extremely Positive')
negative_raw = (df['Sentiment'] == 'Negative') | (df['Sentiment'] == 'Extremely Negative')
df.loc[positive_raw, 'Sentiment'] = 1
df.loc[negative_raw, 'Sentiment'] = 0
df.head(10)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,1
1,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,1
2,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,1
3,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",0
4,3804,48756,"ÃÂT: 36.319708,-82.363649",16-03-2020,As news of the regionÃÂs first confirmed COV...,1
5,3805,48757,"35.926541,-78.753267",16-03-2020,Cashier at grocery store was sharing his insig...,1
6,3807,48759,"Atlanta, GA USA",16-03-2020,Due to COVID-19 our retail store and classroom...,1
7,3808,48760,"BHAVNAGAR,GUJRAT",16-03-2020,"For corona prevention,we should stop to buy th...",0
8,3810,48762,"Pitt Meadows, BC, Canada",16-03-2020,"Due to the Covid-19 situation, we have increas...",1
9,3811,48763,Horningsea,16-03-2020,#horningsea is a caring community. LetÃÂs AL...,1


Сбалансированы ли классы?

In [96]:
number_of_positive_raw = len(df[df['Sentiment'] == 1])
number_of_negative_raw = len(df[df['Sentiment'] == 0])
print(f'Количество положительных строк: {number_of_positive_raw}')
print(f'Количество негативных строк: {number_of_negative_raw}')

Количество положительных строк: 18046
Количество негативных строк: 15398


**Ответ:** Классы можно назвать сбалансированными, количество объектов в обоих классах достаточное большое.


Выведете на экран информацию о пропусках в данных. Если пропуски присутствуют заполните их строкой 'Unknown'.

In [97]:
missing = df.isnull().sum()
missing

,0
UserName,0
ScreenName,0
Location,7049
TweetAt,0
OriginalTweet,0
Sentiment,0


In [98]:
df = df.fillna('Unknown')
df.isnull().sum()

,0
UserName,0
ScreenName,0
Location,0
TweetAt,0
OriginalTweet,0
Sentiment,0


Разделите данные на обучающие и тестовые в соотношении 7 : 3 и укажите `random_state=0`

In [99]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
train, test = train_test_split(df, test_size=0.3, random_state=0)
print(f'Кол-во данных в train: {len(train)}')
print(f'Кол-во данных в test: {len(test)}')

Кол-во данных в train: 23410
Кол-во данных в test: 10034


## Задание 2 Токенизация (3 балла)

Постройте словарь на основе обучающей выборки и посчитайте количество встреч каждого токена с использованием самой простой токенизации - деления текстов по пробельным символам и приведения токенов в нижний регистр.

In [100]:
import warnings
from collections import Counter
warnings.filterwarnings("ignore")

In [101]:
tokens = Counter()
for text in df['OriginalTweet']:
  lower_text = text.lower()
  text_tokens = lower_text.split()
  tokens.update(text_tokens)
tokens = dict(tokens)
tokens

{'advice': 195,
 'talk': 115,
 'to': 33447,
 'your': 3999,
 'neighbours': 24,
 'family': 392,
 'exchange': 25,
 'phone': 114,
 'numbers': 72,
 'create': 117,
 'contact': 271,
 'list': 250,
 'with': 5764,
 'of': 18578,
 'schools': 78,
 'employer': 17,
 'chemist': 19,
 'gp': 9,
 'set': 231,
 'up': 2875,
 'online': 2388,
 'shopping': 2300,
 'accounts': 34,
 'if': 3058,
 'poss': 1,
 'adequate': 30,
 'supplies': 537,
 'regular': 101,
 'meds': 18,
 'but': 3082,
 'not': 3873,
 'over': 1112,
 'order': 485,
 'coronavirus': 1376,
 'australia:': 1,
 'woolworths': 15,
 'give': 404,
 'elderly,': 19,
 'disabled': 69,
 'dedicated': 63,
 'hours': 389,
 'amid': 798,
 'covid-19': 4504,
 'outbreak': 573,
 'https://t.co/binca9vp8p': 1,
 'my': 3557,
 'food': 5409,
 'stock': 1525,
 'is': 10596,
 'the': 38250,
 'only': 1188,
 'one': 1548,
 'which': 729,
 'empty...': 3,
 'please,': 36,
 "don't": 778,
 'panic,': 71,
 'there': 1710,
 'will': 3865,
 'be': 5058,
 'enough': 501,
 'for': 12193,
 'everyone': 854,
 '

Какой размер словаря получился?

In [102]:
len(tokens)

103200

Выведите 10 самых популярных токенов с количеством встреч каждого из них. Объясните, почему именно эти токены в топе.

In [103]:
values = tokens.items()
sorted_tokens = sorted(values, key=lambda x: x[1], reverse=True)
sorted_tokens[:10]

[('the', 38250),
 ('to', 33447),
 ('and', 20935),
 ('of', 18578),
 ('a', 16667),
 ('in', 16024),
 ('for', 12193),
 ('#coronavirus', 11759),
 ('is', 10596),
 ('are', 9958)]

**Ответ:** среди самых популярных токенов можно увидеть различные предлоги и союзы. Они встречаются в текстах чаще всего, но не несут особой смысловой нагрузки. Также встретился хештэг #coronavirus. Это связано с тем, что основная тема твиттов - коронавирус.

Удалите стоп-слова из словаря и выведите новый топ-10 токенов (и количество встреч) по популярности.  Что можно сказать  о нем?

In [107]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
sorted_tokens = dict(sorted_tokens)
for word in stopwords.words("english"):
  if word in sorted_tokens:
    del sorted_tokens[word]
print(f"Количество токенов после удаления стоп-слов: {len(sorted_tokens)}")

Количество токенов после удаления стоп-слов: 103009


In [108]:
popular_tokens = list(sorted_tokens.items())[:10]
popular_tokens

[('#coronavirus', 11759),
 ('prices', 5625),
 ('food', 5409),
 ('grocery', 4882),
 ('supermarket', 4662),
 ('covid-19', 4504),
 ('people', 4488),
 ('store', 4486),
 ('#covid19', 3561),
 ('consumer', 3233)]

**Ответ:** после удаления стоп-слов, список популярных слов больше не содержит предлогов. В списке содержатся слова, напрямую связанные с коронавирусом. Преимущественно слова связаны с продуктовыми магазинами и едой, поскольку в то время это было основной проблемой. Полки магазинов были пустые, люди скупали все. Поэтому в твиттах часто фигурируют слова people, food, prices, grocery, covid-19 и тд.


Также выведите 20 самых непопулярных слов (если самых непопулярных слов больше, выведите любые 20 из них) Почему эти токены непопулярны, требуется ли как-то дополнительно работать с ними?

In [109]:
non_popular_tokens = list(sorted_tokens.items())[:-21:-1]
non_popular_tokens

[('whethe', 1),
 ('$700.00', 1),
 ('rift', 1),
 ('new/used', 1),
 ('@tartiicat', 1),
 ('martinsville,', 1),
 ('@kameronwilds', 1),
 ('rejecting', 1),
 ('delays.', 1),
 ('@mrsilverscott', 1),
 ('https://t.co/v8xdxhqeyn', 1),
 ('home??', 1),
 ('https://t.co/s8coy5vvgn', 1),
 ('gms.', 1),
 ('rs.46,215', 1),
 ('$1,722.20', 1),
 ('dec.', 1),
 ('$1,715.25/ounce', 1),
 ('#safe-haven', 1),
 ("bullion's", 1)]

**Ответ:** среди непопулярных слов встречаются имена пользователей(например, @tartiicat), также слова с ошибками(bullion's, вероятнее подразумевалось billion's), слова со знаками препинания(home??), числа($1,722.20) и ссылки(https://t.co/s8coy5vvgn).

Имена пользователей встречаются редко, поскольку люди могут отмечать своих друзей и тд, которые непопулярны в твиттере. Для лучшей работы с текстом их лучше оставить в таком виде, в котором они присутствуют сейчас - то есть с символом собачки в начале, чтобы модель не распознавала их как осмысленные слова.

Слова с ошибками можно обработать и привести к правильному виду.

Слова со знаками препинания нужно обработать. Вероятнее всего они уже есть в словаре и без знака препинания. Обработка поможет сократить размер словаря.

Числа в целом можно оставить как есть, например, для стоимости стоит оставлять знак доллара перед числом, чтобы модель корректно отделяла цену от года, количества чего-нибудь и тд.






Теперь воспользуемся токенайзером получше - TweetTokenizer из библиотеки nltk. Примените его и посмотрите на топ-10 популярных слов. Чем он отличается от топа, который получался раньше? Почему?

In [110]:
from nltk.tokenize import TweetTokenizer

tweet_tokens = Counter()
tw = TweetTokenizer()
for text in df['OriginalTweet']:
  lower_text = text.lower()
  text_tokens = tw.tokenize(lower_text)
  tweet_tokens.update(text_tokens)
tokens = dict(tweet_tokens)
print(f'Длина словаря с применением TweetTokenazer: {len(tokens)}')

Длина словаря с применением TweetTokenazer: 75011


In [111]:
values = tokens.items()
sorted_tokens = sorted(values, key=lambda x: x[1], reverse=True)
sorted_tokens[:10]

[('the', 38499),
 ('.', 34284),
 ('to', 33588),
 (',', 25142),
 ('and', 21134),
 ('of', 18622),
 ('a', 16863),
 ('in', 16232),
 ('?', 13730),
 ('#coronavirus', 12587)]

**Ответ:** новый топ отличается от предыдущего тем, что помимо предлогов содержит также и знаки препинания ('?', '.', ','). Это происходит благодаря TweetTokenizer(), который отделяет знаки препинания от слов.

Удалите из словаря стоп-слова и пунктуацию, посмотрите на новый топ-10 слов с количеством встреч, есть ли теперь в нем что-то не похожее на слова?

In [112]:
from string import punctuation
noise = stopwords.words("english") + list(punctuation)
sorted_tokens = dict(sorted_tokens)
for word in noise:
  if word in sorted_tokens:
    del sorted_tokens[word]
print(f'Длина словаря после удаления стоп-слов и знаков препинания: {len(sorted_tokens)}')

Длина словаря после удаления стоп-слов и знаков препинания: 74788


In [113]:
popular_tokens = list(sorted_tokens.items())[:10]
popular_tokens

[('#coronavirus', 12587),
 ('â', 10498),
 ('\x82', 10361),
 ('19', 10142),
 ('covid', 8832),
 ('prices', 6644),
 ('food', 6213),
 ('\x92', 6190),
 ('store', 5494),
 ('supermarket', 5435)]

**Ответ:** в новом топе встречаются слова, связанные с продуктами (prices, food, store, supermarket) и коронавирусом (#coronavirus).Covid-19 новый токенайзер разделил на covid и 19.

Помимо осмысленных слов в топе содержатся 'â', '\x82', '\x92'. Первый символ может возникнуть из-за некорректной кодировки. Два других - какие-то обозначения.

Скорее всего в некоторых топах были неотображаемые символы или отдельные буквы не латинского алфавита. Уберем их: удалите из словаря токены из одного символа, позиция которого в таблице Unicode 128 и более (`ord(x) >= 128`)

Выведите топ-10 самых популярных и топ-20 непопулярных слов. Чем полученные топы отличаются от итоговых топов, полученных при использовании токенизации по пробелам? Что теперь лучше, а что хуже?

In [114]:
clean_sorted_tokens = dict()
for word in sorted_tokens:
  if len(word) != 1 or ord(word) < 128:
    clean_sorted_tokens[word] = sorted_tokens[word]
print(f'Длина словаря после удаления отдельных букв не латинского алфавита: {len(clean_sorted_tokens)}')

Длина словаря после удаления отдельных букв не латинского алфавита: 74744


In [115]:
popular_tokens = list(clean_sorted_tokens.items())[:10]
popular_tokens

[('#coronavirus', 12587),
 ('19', 10142),
 ('covid', 8832),
 ('prices', 6644),
 ('food', 6213),
 ('store', 5494),
 ('supermarket', 5435),
 ('grocery', 4959),
 ('people', 4902),
 ('#covid19', 3726)]

In [116]:
non_popular_tokens = list(clean_sorted_tokens.items())[:-21:-1]
non_popular_tokens

[('whethe', 1),
 ('700.00', 1),
 ('rift', 1),
 ('@tartiicat', 1),
 ('martinsville', 1),
 ('@kameronwilds', 1),
 ('rejecting', 1),
 ('@mrsilverscott', 1),
 ('https://t.co/v8xdxhqeyn', 1),
 ('https://t.co/s8coy5vvgn', 1),
 ('gms', 1),
 ('46,215', 1),
 ('1,722', 1),
 ('1,715', 1),
 ('#safe-haven', 1),
 ("bullion's", 1),
 ('@majangchien', 1),
 ('https://t.co/quq8y0um6n', 1),
 ('https://t.co/g0ri0egp6m', 1),
 ('jaffri', 1)]

**Ответ:** списки самых популярных 10 слов практически совпадают для рассмотренных способов токенизации. Отличаются они только словом covid-19: токенизация по пробелам считает его за одно слово, а TweetTokenizer - разделяет это слово на два: covid и 19.

Списки непопулярных слов имеют отличия. Новый топ больше не содержит слов со знаками препинания(в первом топе - 'home??', в новом топе это слово отсутствует). Также рядом с числами пропали символы и сокращения, такие как $1,715.25/ounce.

Таким образом, второй токенизатор позволил уменьшить размер словаря, путем отделения и удаления знаков препинания, а также некорректных символов, что очень хорошо.

Из минусов: мне кажется, что для чисел, которые обозначают стоимость, лучше оставлять знак доллара в начале, как делал это первый токенизатор.

Выведите топ-10 популярных хештегов (токены, первые символы которых - #) с количеством встреч. Что можно сказать о них?

In [117]:
hashtags = {word: value for word, value in clean_sorted_tokens.items() if '#' in word}
list(hashtags.items())[:10]

[('#coronavirus', 12587),
 ('#covid19', 3726),
 ('#covid_19', 2525),
 ('#covid2019', 1370),
 ('#toiletpaper', 1070),
 ('#covid', 919),
 ('#socialdistancing', 701),
 ('#coronacrisis', 627),
 ('#pandemic', 359),
 ('#coronaviruspandemic', 344)]

**Ответ:** в топ входят хештэги, включающие слова corona, covid, pandemic - слова, связанные непосредственно с коронавирусом. Также есть хештэг #toiletpaper - во времена коронавируса остро стоял вопрос по поводу туалетной бумаги, поскольку люди скупали ее в огромных количествах, думая, что долго просидят в своих квартирах без возможности выйти на улицу.

Таким образом, наиболее популярные хештэги соответствуют основной теме твиттов.

То же самое проделайте для ссылок на сайт https://t.co Сравнима ли популярность ссылок с популярностью хештегов? Будет ли информация о ссылке на конкретную страницу полезна?

In [118]:
links = {word: value for word, value in clean_sorted_tokens.items() if 'https://t.co' in word}
list(links.items())[:10]

[('https://t.co/oxa7swtond', 6),
 ('https://t.co/g63rp042ho', 5),
 ('https://t.co/r7sagojsjg', 4),
 ('https://t.co/wrlhyzizaa', 4),
 ('https://t.co/ymsemlvttd', 4),
 ('https://t.co/3kfuiojxep', 4),
 ('https://t.co/oi39zsanq8', 4),
 ('https://t.co/6yvykiab2c', 4),
 ('https://t.co/xpcm2xkj4o', 4),
 ('https://t.co/gu6b4xpqp4', 4)]

**Ответ:** популярность ссылок не сравнима с популярностью хештэгов: хештэги встречаются в сотни, а то и в тысячи раз больше. Поэтому информация о ссылке на конкретную страницу будет мало полезна.

Используем опыт предыдущих экспериментов и напишем собственный токенайзер, улучшив TweetTokenizer. Функция tokenize должна:



*   Привести текст в нижний регистр
*   Применить TweetTokenizer для  выделения токенов
*   Удалить стоп-слова, пунктуацию, токены из одного символа с позицией в таблице Unicode 128 и более,  ссылки на t.co



In [119]:
def custom_tokenizer(text):
  lower_text = text.lower()

  text_tokens = tw.tokenize(lower_text)

  one_symbol_tokens = {word for word in text_tokens if len(word) == 1 and ord(word) >= 128}
  links = {word for word in text_tokens if 'https://t.co' in word}
  noise = stopwords.words("english") + list(punctuation) + list(one_symbol_tokens) + list(links)

  tokens = [word for word in text_tokens if word not in noise]

  return tokens


In [120]:
custom_tokenizer('This is sample text!!!! @Sample_text I, \x92\x92 https://t.co/sample  #sampletext')

['sample', 'text', '@sample_text', '#sampletext']

## Задание 3 Векторизация текстов (2 балла)

Обучите CountVectorizer с использованием custom_tokenizer в качестве токенайзера. Как размер полученного словаря соотносится с размером изначального словаря из начала задания 2?

In [123]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(tokenizer=custom_tokenizer)
cv.fit(df['OriginalTweet'])
print(len(cv.vocabulary_))

56520


**Ответ:** Размер изначального словаря - 103200. Размер словаря с использованием кастомного токенайзера - 56520. Получили словарь в два раза меньше, чем первоначально, что является хорошим результатом.

Посмотрим на какой-нибудь конкретный твитт:

In [124]:
ind = 9023
train.iloc[ind]['OriginalTweet'], train.iloc[ind]['Sentiment']

('Nice one @SkyNews lets not panic but show ppl in france queueing for food!!! #CoronavirusOutbreak #COVID2019 brainless!! Ffs',
 np.int64(0))

Автор твитта не доволен ситуацией с едой во Франции и текст имеет резко негативную окраску.

Примените обученный CountVectorizer для векторизации данного текста, и попытайтесь определить самый важный токен и самый неважный токен (токен, компонента которого в векторе максимальна/минимальна, без учета 0). Хорошо ли они определились, почему?

In [125]:
tweet = train.iloc[ind]['OriginalTweet']
vector = cv.transform([tweet]).toarray()
tweet_vector = pd.DataFrame(vector, columns=cv.get_feature_names_out())
tweet_vector_tokens = tweet_vector.loc[:, (tweet_vector != 0).any(axis=0)]
tweet_vector_tokens

,#coronavirusoutbreak,#covid2019,@skynews,brainless,ffs,food,france,lets,nice,one,panic,ppl,queueing,show
0,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [130]:
max_count = tweet_vector_tokens.values.max()
min_count = tweet_vector_tokens.values.min()
max_token = tweet_vector_tokens.columns[tweet_vector_tokens.eq(max_count).any()]
min_token = tweet_vector_tokens.columns[tweet_vector_tokens.eq(min_count).any()]
print(f'Самый важный токен {list(max_token)} - встречается {max_count} раз')
print(f'Самый неважный токен {list(min_token)} - встречается {min_count}  раз')

Самый важный токен ['#coronavirusoutbreak', '#covid2019', '@skynews', 'brainless', 'ffs', 'food', 'france', 'lets', 'nice', 'one', 'panic', 'ppl', 'queueing', 'show'] - встречается 1 раз
Самый неважный токен ['#coronavirusoutbreak', '#covid2019', '@skynews', 'brainless', 'ffs', 'food', 'france', 'lets', 'nice', 'one', 'panic', 'ppl', 'queueing', 'show'] - встречается 1  раз


**Ответ:** самый важный и неважный токен в данном случае определились плохо, потому что каждое слово встречается в тексте один раз. Для того чтобы понять, какое из слов важнее, имеется мало информации.

Теперь примените TfidfVectorizer и  определите самый важный/неважный токены. Хорошо ли определились, почему?

In [131]:
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(tokenizer=custom_tokenizer)
tv.fit(df['OriginalTweet'])
print(len(tv.vocabulary_))

56520


In [132]:
vector_tv = tv.transform([tweet]).toarray()
tweet_vector_tv = pd.DataFrame(vector_tv, columns=tv.get_feature_names_out())
tweet_vector_tokens_tv = tweet_vector_tv.loc[:, (tweet_vector_tv != 0).any(axis=0)]
tweet_vector_tokens_tv

,#coronavirusoutbreak,#covid2019,@skynews,brainless,ffs,food,france,lets,nice,one,panic,ppl,queueing,show
0,0.22274,0.165977,0.31456,0.396818,0.322774,0.112289,0.31456,0.301963,0.250648,0.16127,0.146218,0.25215,0.356801,0.241674


In [135]:
max_count_tv = tweet_vector_tokens_tv.values.max()
min_count_tv = tweet_vector_tokens_tv.values.min()
max_token_tv = tweet_vector_tokens_tv.columns[tweet_vector_tokens_tv.eq(max_count_tv).any()]
min_token_tv = tweet_vector_tokens_tv.columns[tweet_vector_tokens_tv.eq(min_count_tv).any()]
print(f'Самый важный токен {list(max_token_tv)} - коэффициент TF-IDF: {max_count_tv}')
print(f'Самый неважный токен {list(min_token_tv)} - коэффициент TF-IDF: {min_count_tv}')

Самый важный токен ['brainless'] - коэффициент TF-IDF: 0.3968179410547698
Самый неважный токен ['food'] - коэффициент TF-IDF: 0.11228923948408494


**Ответ:** при такой токенизации выделить самый важный и неважный токен получается лучше, поскольку каждое слово имеет свой вес. Вес токена складывается из частоты встречаемости слова в данном конкретном тексте и частоты встречаемости в остальных: "чем чаще данное слово встречается в данном тексте и чем реже в остальных, тем важнее оно для этого текста"(из семинара https://github.com/hse-ds/iad-intro-ds/blob/master/2025/seminars/11_texts/sem11_texts.ipynb)

Такой подход позволяет для каждого слова определить его важность.

В данном случае самым важным словом является brainless - безмозглый. Оно несет в себе сильную негативную оценку.

Самым неважным является food - еда. Оно не несет в себе никакой оценки.


Найдите какой-нибудь положительно окрашенный твитт, где TfidfVectorizer хорошо (полезно для определения окраски) выделяет важный токен, поясните пример.

*Подсказка:* явно положительные твитты можно искать при помощи положительных слов (good, great, amazing и т. д.)

In [136]:
train[train['OriginalTweet'].apply(lambda x: 'amazing' in x) & (train['Sentiment'] == 1)]

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
4583,9362,54314,"Moulton, England",19-03-2020,Hearing so many stories of NHS heroes Teachers...,1
8221,13787,58739,Unknown,20-03-2020,Let s just take a minute to say THANK YOU also...,1
3183,7654,52606,"London, England",18-03-2020,"Back at the ""Frontline""\r\r\nA massive shout o...",1
24347,33577,78529,Unknown,05-04-2020,Massive thanks to @waitrose for my delivery of...,1
3281,7772,52724,"Nairobi, Kenya",18-03-2020,Crisp clean fresh air perfect ambience Covid 1...,1
...,...,...,...,...,...,...
8199,13757,58709,Wrightington,20-03-2020,The support from customers this week has been ...,1
11636,17911,62863,Australia,21-03-2020,"Margot Robbie is an amazing actress, and love ...",1
23018,31918,76870,"Karachi, Pakistan",04-04-2020,Face Mask (Pack of 5) ÃÂ Meeting the need of...,1
5208,10126,55078,Unknown,19-03-2020,There's some amazing work going on in the worl...,1


In [137]:
ind = 8221
train.loc[ind]['OriginalTweet']

'Let s just take a minute to say THANK YOU also to the amazing postal workers So many real people behind a click of online shopping and getting cards delivered Thank   you   19'

In [138]:
positive_tweet = train.loc[ind]['OriginalTweet']
vector_tv = tv.transform([positive_tweet]).toarray()
tweet_vector_tv = pd.DataFrame(vector_tv, columns=tv.get_feature_names_out())
tweet_vector_tokens_tv = tweet_vector_tv.loc[:, (tweet_vector_tv != 0).any(axis=0)]
tweet_vector_tokens_tv

,19,also,amazing,behind,cards,click,delivered,getting,let,many,minute,online,people,postal,real,say,shopping,take,thank,workers
0,0.086939,0.179743,0.251806,0.254853,0.282009,0.255727,0.255142,0.197294,0.214187,0.170908,0.29648,0.141871,0.119671,0.294847,0.218496,0.200254,0.139984,0.177202,0.369407,0.152187


In [140]:
max_count_tv = tweet_vector_tokens_tv.values.max()
min_count_tv = tweet_vector_tokens_tv.values.min()
max_token_tv = tweet_vector_tokens_tv.columns[tweet_vector_tokens_tv.eq(max_count_tv).any()]
min_token_tv = tweet_vector_tokens_tv.columns[tweet_vector_tokens_tv.eq(min_count_tv).any()]
print(f'Самый важный токен {list(max_token_tv)} - коэффициент TF-IDF: {max_count_tv}')
print(f'Самый неважный токен {list(min_token_tv)} - коэффициент TF-IDF: {min_count_tv}')

Самый важный токен ['thank'] - коэффициент TF-IDF: 0.369406977658367
Самый неважный токен ['19'] - коэффициент TF-IDF: 0.0869393606759461


**Ответ:** твит содержит благодарность почтовым работникам и имеет положительную окраску.

Наиболее важным словом в твитте является слово 'thank' - спасибо. Это говорит о том, что данное слово не так часто встречается в текстах, поэтому его наличие в тексте может указывать на положительную окраску твитта.

Неважным словом является '19'. Оно не несет в себе никакой окраски. В текстах про коронавирус число 19 - наиболее распространенное, поскольку это год начала распространения болезни. Из-за того, что оно так часто встречается, его важность получается очень низкой.

## Задание 4 Обучение первых моделей (1 балл)

Примените оба векторайзера для получения матриц с признаками текстов.  Выделите целевую переменную.

In [141]:
y_train = train['Sentiment']
x_train = train.drop('Sentiment', axis=1)

In [142]:
y_test = test['Sentiment']
x_test = test.drop('Sentiment', axis=1)

In [143]:
cv2 = CountVectorizer(tokenizer=custom_tokenizer)
x_train_transform_cv = cv2.fit_transform(x_train['OriginalTweet'])
x_test_transform_cv = cv2.transform(x_test['OriginalTweet'])

In [144]:
tv2 = TfidfVectorizer(tokenizer=custom_tokenizer)
x_train_transform_tv = tv2.fit_transform(x_train['OriginalTweet'])
x_test_transform_tv = tv2.transform(x_test['OriginalTweet'])

Обучите логистическую регрессию на векторах из обоих векторайзеров. Посчитайте долю правильных ответов на обучающих и тестовых данных. Какой векторайзер показал лучший результат? Что можно сказать о моделях?

Используйте `sparse` матрицы (после векторизации), не превращайте их в `numpy.ndarray` или `pd.DataFrame` - может не хватить памяти.

In [145]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf_cv = LogisticRegression()
clf_cv.fit(x_train_transform_cv, y_train)

y_train_pred = clf_cv.predict(x_train_transform_cv)
y_test_pred = clf_cv.predict(x_test_transform_cv)

accuracy_train_cv = accuracy_score(y_train, y_train_pred)
accuracy_test_cv = accuracy_score(y_test, y_test_pred)

print(f"Доля верных ответов на train: {accuracy_train_cv}")
print(f"Доля верных ответов на test: {accuracy_test_cv}")

Доля верных ответов на train: 0.9847073900042717
Доля верных ответов на test: 0.8673510065776361


In [146]:
clf_tv = LogisticRegression()
clf_tv.fit(x_train_transform_cv, y_train)

y_train_pred = clf_tv.predict(x_train_transform_tv)
y_test_pred = clf_tv.predict(x_test_transform_tv)

accuracy_train_tv = accuracy_score(y_train, y_train_pred)
accuracy_test_tv = accuracy_score(y_test, y_test_pred)

print(f"Доля верных ответов на train: {accuracy_train_tv}")
print(f"Доля верных ответов на test: {accuracy_test_tv}")

Доля верных ответов на train: 0.9833404527979496
Доля верных ответов на test: 0.8508072553318716


**Ответ:** в данном случае на обучающих данных оба векторайзера дали одинаково хороший результат.

В случае с тестовыми данными CountVectorizer на 0.01 лучше, чем TfidfVectorizer, можно говорить, что у первого векторайзер обобщающая способность лучше.

Если говорить в целом, то обе модели достаточно хороши и имеют высокие доли верных ответов на тестовых подвыборках.

TfidfVectorizer, не смотря на то, что он показал результат хуже, все-таки позволяет учитывать, насколько токен общеупотребляемый и несет ли он в себе что-то значимое для определения тональности текста. Поэтому, мне кажется, что лучше использовать его.




## Задание 5 Стемминг (0.5 балла)

Для уменьшения словаря можно использовать стемминг.

Модифицируйте написанный токенайзер, добавив в него стемминг с использованием SnowballStemmer. Обучите Count- и Tfidf- векторайзеры. Как изменился размер словаря?

In [147]:
from nltk.stem.snowball import SnowballStemmer

In [148]:
stemmer = SnowballStemmer("english")
def custom_stem_tokenizer(text):
  lower_text = text.lower()

  text_tokens = tw.tokenize(lower_text)

  one_symbol_tokens = {word for word in text_tokens if len(word) == 1 and ord(word) >= 128}
  links = {word for word in text_tokens if 'https://t.co' in word}
  noise = stopwords.words("english") + list(punctuation) + list(one_symbol_tokens) + list(links)

  clean_tokens = [word for word in text_tokens if word not in noise]

  tokens = [stemmer.stem(w) for w in clean_tokens]

  return tokens

In [149]:
custom_stem_tokenizer('This is sample text!!!! @Sample_text I, \x92\x92 https://t.co/sample  #sampletext adding more words to check stemming')

['sampl', 'text', '@sample_text', '#sampletext', 'ad', 'word', 'check', 'stem']

In [150]:
cv = CountVectorizer(tokenizer=custom_stem_tokenizer)
x_train_transform_cv = cv.fit_transform(x_train['OriginalTweet'])
x_test_transform_cv = cv.transform(x_test['OriginalTweet'])
print(len(cv.vocabulary_))

36632


In [152]:
tv = TfidfVectorizer(tokenizer=custom_stem_tokenizer)
x_train_transform_tv = tv.fit_transform(x_train['OriginalTweet'])
x_test_transform_tv = tv.transform(x_test['OriginalTweet'])
print(len(tv.vocabulary_))

36632


**Ответ:** размер словаря сократился на одну треть, это довольно хороший результат.

Обучите логистическую регрессию с использованием обоих векторайзеров. Изменилось ли качество? Есть ли смысл применять стемминг?

In [153]:
clf = LogisticRegression()
clf.fit(x_train_transform_cv, y_train)

y_train_pred = clf.predict(x_train_transform_cv)
y_test_pred = clf.predict(x_test_transform_cv)

accuracy_train_cv = accuracy_score(y_train, y_train_pred)
accuracy_test_cv = accuracy_score(y_test, y_test_pred)

print(f"Доля верных ответов на train: {accuracy_train_cv}")
print(f"Доля верных ответов на test: {accuracy_test_cv}")

Доля верных ответов на train: 0.9720632208457924
Доля верных ответов на test: 0.8671516842734702


In [154]:
clf = LogisticRegression()
clf.fit(x_train_transform_cv, y_train)

y_train_pred = clf.predict(x_train_transform_tv)
y_test_pred = clf.predict(x_test_transform_tv)

accuracy_train_tv = accuracy_score(y_train, y_train_pred)
accuracy_test_tv = accuracy_score(y_test, y_test_pred)

print(f"Доля верных ответов на train: {accuracy_train_tv}")
print(f"Доля верных ответов на test: {accuracy_test_tv}")

Доля верных ответов на train: 0.976420333190944
Доля верных ответов на test: 0.8572852302172613


**Ответ:** качество не изменилось. Для данной задачи можно не применять стемминг, но он может быть полезен для сокращения количества токенов. Это позволит быстрее обрабатывать данные.

## Задание  6 Работа с частотами (1.5 балла)

Еще один способ уменьшить количество признаков - это использовать параметры min_df и max_df при построении векторайзера  эти параметры помогают ограничить требуемую частоту встречаемости токена в документах.

По умолчанию берутся все токены, которые встретились хотя бы один раз.



Подберите max_df такой, что размер словаря будет 36651 (на 1 меньше, чем было). Почему параметр получился такой большой/маленький?

In [155]:
cv_df = CountVectorizer(tokenizer=custom_stem_tokenizer,
                        max_df=8000
                        ).fit(x_train['OriginalTweet'])
print(len(cv_df.vocabulary_))

36631


**Ответ:** параметр max_df = 8000 позволил снизить количество токенов  в словаре на один. Это достаточно большое значение, если перевести его в долю, то получится где-то одна треть от общего количества записей в датасете.

Параметр позволяет добавлять в словарь только те токены, которые встретились не бьльше, чем в 8000 твиттов и исключить популярные слова, которые вероятнее всего не несут никакой значимости для определения тональности текста.

Большое значение параметра max_df может говорить о большом числе таких общеупотребляемых слов.

Подберите min_df (используйте дефолтное значение max_df) в CountVectorizer таким образом, чтобы размер словаря был 3700 токенов (при использовании токенайзера со стеммингом), а качество осталось таким же, как и было. Что можно сказать о результатах?

In [159]:
cv_df = CountVectorizer(tokenizer=custom_stem_tokenizer,
                        min_df=11
                        ).fit(x_train['OriginalTweet'])
print(len(cv_df.vocabulary_))

3687


In [160]:
x_train_transform_cv = cv_df.fit_transform(x_train['OriginalTweet'])
x_test_transform_cv = cv_df.transform(x_test['OriginalTweet'])

clf_cv2 = LogisticRegression()
clf_cv2.fit(x_train_transform_cv, y_train)

y_train_pred = clf_cv2.predict(x_train_transform_cv)
y_test_pred = clf_cv2.predict(x_test_transform_cv)

accuracy_train_cv = accuracy_score(y_train, y_train_pred)
accuracy_test_cv = accuracy_score(y_test, y_test_pred)

print(f"Доля верных ответов на train: {accuracy_train_cv}")
print(f"Доля верных ответов на test: {accuracy_test_cv}")

Доля верных ответов на train: 0.9290046988466467
Доля верных ответов на test: 0.8680486346422165


**Ответ:** подобрать параметр, при котором токенов будет ровно 3700 не получилось. Наиболее близкое число - 3687.

Парметр min_df = 11 позволяет отбросить те слова, которые встречаются менее, чем в 11 твиттах. Качество модели на обучающей выборке при min_df = 11 уменьшилось на 0.4, но на тестовых данных повысилось на 0.01, поэтому можно сказать, что качество модели не ухудшилось.

В предыдущих заданиях признаки не скалировались. Отскалируйте данные (при словаре размера 3.7 тысяч, векторизованные CountVectorizer), обучите логистическую регрессию, посмотрите качество и выведите `barplot`, содержащий по 10 токенов, с наибольшим по модулю положительными/отрицательными весами. Что можно сказать об этих токенах?

In [ ]:
from sklearn.preprocessing import StandardScaler
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

## Задание 7 Другие признаки (1.5 балла)

Мы были сконцентрированы на работе с текстами твиттов и не использовали другие признаки - имена пользователя, дату и местоположение

Изучите признаки UserName и ScreenName. полезны ли они? Если полезны, то закодируйте их, добавьте к матрице с отскалированными признаками, обучите логистическую регрессию, замерьте качество.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Изучите признак TweetAt в обучающей выборке: преобразуйте его к типу datetime и нарисуйте его гистограмму с разделением по цвету на основе целевой переменной. Полезен ли он? Если полезен, то закодируйте его, добавьте к матрице с отскалированными признаками, обучите логистическую регрессию, замерьте качество.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --



Поработайте с признаком Location в обучающей выборке. Сколько уникальных значений?

In [ ]:
# -- YOUR CODE HERE --

Постройте гистограмму топ-10 по популярности местоположений (исключая Unknown)

In [ ]:
# -- YOUR CODE HERE --

Видно, что многие местоположения включают в себя более точное название места, чем другие (Например, у некоторых стоит London, UK; а у некоторых просто UK или United Kingdom).

Создайте новый признак WiderLocation, который содержит самое широкое местоположение (например, из London, UK должно получиться UK). Сколько уникальных категорий теперь? Постройте аналогичную гистограмму.

In [ ]:
# -- YOUR CODE HERE --

Закодируйте признак WiderLocation с помощью OHE таким образом, чтобы создались только столбцы для местоположений, которые встречаются более одного раза. Сколько таких значений?


In [ ]:
# -- YOUR CODE HERE --

Добавьте этот признак к матрице отскалированных текстовых признаков, обучите логистическую регрессию, замерьте качество. Как оно изменилось? Оказался ли признак полезным?


*Подсказка:* используйте параметр `categories` в энкодере.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --